# 11 — Model vs Market Overlay

For completed games, overlay our model's P(win) on the Kalshi trade price chart. Visual sanity check: does the model track the market?

In [ ]:
import sys
sys.path.insert(0, '/Users/chriswang/Desktop/prediction-market-exploration/nba-edge')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from nba_edge.data.s3_reader import discover_trade_dates, load_trades, load_boxscores_for_game, discover_game_ids
from nba_edge.data.ticker_parser import parse_ticker, is_game_winner
from nba_edge.data.game_alignment import align_trades_with_game
from nba_edge.models.analytical import AnalyticalWinProb, parse_game_clock

In [ ]:
# Load trades and find GAME market tickers
dates = discover_trade_dates(min_date='2026-04-21')
df = load_trades(dates)

game_tickers = [t for t in df['market_ticker'].unique().to_list() if is_game_winner(t)]
print(f'GAME tickers: {len(game_tickers)}')
for t in sorted(game_tickers):
    p = parse_ticker(t)
    n = df.filter(pl.col('market_ticker') == t).height
    print(f'  {t:45s} ({n} trades)')

In [ ]:
# Pick a ticker with decent volume
# Sort by trade count, pick the most active
ticker_volumes = [(t, df.filter(pl.col('market_ticker') == t).height) for t in game_tickers]
ticker_volumes.sort(key=lambda x: -x[1])
chosen_ticker = ticker_volumes[0][0]
print(f'Chosen: {chosen_ticker} ({ticker_volumes[0][1]} trades)')

In [ ]:
# Parse ticker and find corresponding game
from nba_edge.data.game_alignment import match_ticker_to_game

parsed = parse_ticker(chosen_ticker)
print(f'Series: {parsed.series}, Date: {parsed.game_date}')
print(f'Teams: {parsed.away_team} @ {parsed.home_team}, Selection (YES team): {parsed.selection}')

# Find game_id
available_games = discover_game_ids(parsed.game_date)
print(f'Available games: {len(available_games)}')
matched_id = match_ticker_to_game(chosen_ticker, available_games)

if matched_id:
    matched_game = next(g for g in available_games if g['game_id'] == matched_id)
    print(f"\nMatched game: {matched_game['game_id']} — {matched_game['game_code']}")
else:
    print(f'No match found! Available: {[(g["home_team"], g["away_team"]) for g in available_games]}')

In [ ]:
# Load boxscore snapshots for this game
snapshots = load_boxscores_for_game(matched_game['game_id'], parsed.game_date)
print(f'Boxscore snapshots: {len(snapshots)}')
print(f"Home: {snapshots[0]['home_team']}, Away: {snapshots[0]['away_team']}")
print(f"Final: {snapshots[-1]['home_score']}-{snapshots[-1]['away_score']}")

In [ ]:
# Compute model probability at each boxscore snapshot
model = AnalyticalWinProb(variance_rate=0.44)  # use calibrated value from notebook 10

yes_is_home = (parsed.selection == snapshots[0]['home_team'])

model_times = []  # minutes from first snapshot
model_probs = []  # P(yes team wins)
t0 = snapshots[0]['t_receipt']

for s in snapshots:
    secs = parse_game_clock(s['period'], s['game_clock'])
    diff = s['home_score'] - s['away_score']
    p_home = model.predict(diff, secs)
    p_yes = p_home if yes_is_home else (1.0 - p_home)
    
    model_times.append((s['t_receipt'] - t0) / 60)
    model_probs.append(p_yes * 100)  # convert to cents for comparison

print(f'Model predictions computed: {len(model_probs)}')

In [ ]:
# Get trades for this ticker
ticker_trades = df.filter(pl.col('market_ticker') == chosen_ticker).sort('t_receipt_ns')
trade_times = (ticker_trades['t_receipt_ns'].cast(pl.Float64) / 1e9 - t0).to_numpy() / 60
trade_prices = ticker_trades['price'].to_numpy()

# Plot overlay
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(trade_times, trade_prices, linewidth=1, color='steelblue', alpha=0.8, label='Market Price (cents)')
ax.plot(model_times, model_probs, linewidth=1.5, color='red', alpha=0.7, label='Model P(yes) × 100')

ax.set_xlabel('Minutes from game start')
ax.set_ylabel('Price / Probability (cents)')
ax.set_ylim(0, 100)
ax.axhline(50, color='black', linestyle=':', alpha=0.3)
ax.set_title(f'{chosen_ticker}\n{parsed.away_team} @ {parsed.home_team} — Model vs Market')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# Edge over time: model - market
# For each trade, find the closest model prediction
from nba_edge.data.game_alignment import align_trades_with_game

aligned = align_trades_with_game(
    ticker_trades, snapshots, parsed.selection, snapshots[0]['home_team']
)

edges = []
trade_mins = []
for a in aligned:
    secs = a.seconds_remaining
    diff = a.score_diff
    p_home = model.predict(diff, secs)
    p_yes = p_home if yes_is_home else (1.0 - p_home)
    edge = p_yes - a.trade_price / 100.0
    edges.append(edge * 100)  # in cents
    trade_mins.append((a.t_receipt_ns / 1e9 - t0) / 60)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(trade_mins, edges, linewidth=0.5, color='green', alpha=0.7)
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax.axhline(5, color='red', linestyle='--', alpha=0.5, label='+5c edge threshold')
ax.axhline(-5, color='red', linestyle='--', alpha=0.5, label='-5c edge threshold')
ax.fill_between(trade_mins, -5, 5, alpha=0.05, color='gray')
ax.set_xlabel('Minutes from game start')
ax.set_ylabel('Edge (model - market, cents)')
ax.set_title('Model Edge Over Time')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean edge: {np.mean(edges):.2f}c')
print(f'Std edge:  {np.std(edges):.2f}c')
print(f'Trades with |edge| > 5c: {sum(1 for e in edges if abs(e) > 5)} / {len(edges)}')